# 03 · Hyper-parameter tuning — GPU models (XGBoost, MLP, GraphSAGE, GATv2, ChebNet, GIN)

This notebook chooses the hyper-parameters of the models that run on the GPU, for each feature menu of step 02. Its twin `03-tuning-cpu.ipynb` does the same for Logistic Regression and Random Forest. Both write into the same folder, so they can run one after the other or, if Colab allows two sessions, at the same time.

**Runtime: GPU** (`Runtime → Change runtime type → T4 GPU`). Expected time: about **4.5 hours** (between 3.5 and 7, depending on how many trials the pruning stops early), so one or two Colab sessions. GATv2 is about half of it.

| Model | Menus | Trials per menu | Device |
|---|---|---|---|
| XGBoost | the 6 menus with features | 40 | GPU (2–7 times faster than the CPU, timing test) |
| MLP | the 6 menus with features | 40 | GPU |
| GraphSAGE | the 6 menus with features | 40 | GPU |
| GATv2 | the 6 menus with features | 40 | GPU |
| ChebNet | the 6 menus with features + `none` | 40 (20 on `none`) | GPU |
| GIN | `none` | 20 | GPU |

That is 32 pairs and 1,240 trials.

**Neural models** (MLP and GNNs), as fixed in `config.json`:
- full batch, Adam, binary cross-entropy weighted by $w_+ = \#\text{licit} / \#\text{illicit} \approx 8.06$;
- **early stopping**: after every epoch the PR-AUC on validation; the training stops after 50 (MLP) or 100 (GNN) epochs without improvement, at most 1,000 epochs, and the trial is worth its best epoch. If the best trial of a pair stops at the cap of 1,000 epochs, the pair is tuned again with 2,000;
- **pruning** (Optuna's median rule): from epoch 50 on, a trial is stopped if its best PR-AUC so far is below the median of the earlier trials at the same epoch. A hopeless trial costs a fraction of a full one;
- **graph per period**: the GNNs train on the graph of time steps 1–30 (unknown nodes included, edges made undirected) and are evaluated on the graph of time steps 31–40. There is no edge between time steps, so nothing leaks from validation into training. The MLP sees only the evaluation rows.

**How a pair is tuned** (the same rules for every menu, so that the menus are compared fairly):
- **one tuning per (menu, model) pair**, with the same search space, the same number of trials and the same metric for every menu;
- **objective**: the average precision (PR-AUC) on the evaluation rows of validation (time steps 31–40), with the model trained on the evaluation rows of the train (time steps 1–30). The test set (41–49) is never read here:
$$\mathrm{AP} = \sum_k \left(R_k - R_{k-1}\right) P_k,$$
where $P_k$ and $R_k$ are precision and recall when the $k$ transactions with the highest scores are flagged;
- **trials** chosen by Optuna's TPE sampler: the first 10 at random, then concentrated where the good trials are. The search spaces and the numbers of trials are in `config.json`;
- **seed**: `set_seed(42)` before every trial (Python, NumPy, PyTorch, cuDNN) and `random_state = 42` in every model, so that two trials differ only by their hyper-parameters.

**Resume.** A finished pair has its file `best/<menu>__<model>.json` and is skipped. During a pair, the Optuna study (an SQLite file) is copied to Drive at most every minute and at the end: if Colab stops, run the notebook again and it restarts from the last copy, losing at most a minute of trials. A trial interrupted by a crash is marked as failed and does not count. After a restart the sampler takes the seed 42 + the number of trials already finished, so an interrupted run is reproducible too. A trial that fails (for example, GPU out of memory) is recorded with its error and the study goes on; five failures in a row stop the notebook.

**Output** (in `DATA_PATH/03-tuning/`, shared with the other tuning notebook):

| File | Content |
|---|---|
| `best/<menu>__<model>.json` | best hyper-parameters, their validation PR-AUC, number of trials, code version: read by step 04 |
| `trials/<menu>__<model>.csv` | every trial with its hyper-parameters, value, state and duration |
| `studies/<menu>__<model>.db` | the Optuna study (SQLite) |
| `logs/03-tuning-gpu.log` | the log lines of this notebook |
| `summary.csv` | one row per finished pair, of both tuning notebooks |

The validation PR-AUC of the tuning is optimistic (it is the best of many trials): the results of the project come from the final runs of step 04, retrained with three new seeds.

**Shared code.** The cells marked `SHARED CODE` are identical copies of the code of the other notebooks of steps 03 and 04, so that tuning and final runs use exactly the same data preparation, models and metrics. Their version (`SHARED_CODE_VERSION`) is written in every result, and step 04 checks it.

## 0. Setup

In [1]:
# Install the libraries that Colab does not have by default (only if they are missing).
import importlib.util
import subprocess
import sys

for package, module in [('optuna', 'optuna'), ('torch_geometric', 'torch_geometric')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [2]:
# Connect Google Drive: every input and output of the project lives in DATA_PATH.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/UNIVERSITA/block-crypto/data')
INPUT_DIR = DATA_PATH / '02-feature-menus'      # output of step 02
TUNING_DIR = DATA_PATH / '03-tuning'            # output of the tuning notebooks
LOCAL_INPUT = Path('/content/input')            # local copy of the input files
LOCAL_STUDIES = Path('/content/studies')        # the Optuna studies while they run (copied to Drive)

Mounted at /content/drive


In [3]:
# Copy the inputs from Drive to the Colab disk: reading a local file is much faster.
import shutil

LOCAL_INPUT.mkdir(exist_ok=True)
for name in ['config.json', 'dataset.parquet', 'edges.parquet']:
    source, target = INPUT_DIR / name, LOCAL_INPUT / name
    assert source.exists(), f'{source} is missing: run step 02 first'
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy(source, target)
    print(f'{name}: {target.stat().st_size / 1e6:,.1f} MB')

config.json: 0.1 MB
dataset.parquet: 76.1 MB
edges.parquet: 2.2 MB


In [4]:
NOTEBOOK = 'gpu'
MODELS = ['xgb', 'mlp', 'sage', 'gatv2', 'cheb', 'gin']      # the models tuned by this notebook
ONLY_PAIRS = None    # None = every pair; or a list such as ['graph__xgb'] to run only some pairs
SYNC_SECONDS = 60    # copy the running study to Drive at most this often
MAX_FAILED_IN_A_ROW = 5

# True = delete the tuning results of this notebook's models on Drive and tune them again.
FORCE_RECOMPUTE = False

## 1. Shared code

Data, seeds and metrics (1/4), classical models (2/4), neural models (3/4) and the tuning driver (4/4).

In [5]:
# ==== SHARED CODE 1/4 · data, seeds, metrics · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import json
import os
import random
import sys
import time
from datetime import datetime, timezone
from types import SimpleNamespace

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # deterministic cuBLAS: must be set before PyTorch uses the GPU
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

SHARED_CODE_VERSION = 1
CONFIG_VERSION = 1                 # the version of config.json this code was written for
DETERMINISTIC_ALGORITHMS = True    # ask PyTorch for deterministic GPU kernels (only a warning where none exists)


def set_seed(seed):
    """Fix every random generator: Python, NumPy and, when it is loaded, PyTorch (CPU, every GPU, cuDNN)."""
    random.seed(seed)
    np.random.seed(seed)
    if 'torch' in sys.modules:
        import torch
        torch.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(DETERMINISTIC_ALGORITHMS, warn_only=True)


def atomic(path, write):
    """Write through a temporary file, then rename it: a run that stops while writing
    never leaves a half-written file on Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    write(tmp)
    os.replace(tmp, path)


def to_builtin(value):
    """Make NumPy numbers JSON-serializable."""
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'not JSON-serializable: {type(value)}')


def write_csv(df, path):
    atomic(path, lambda tmp: df.to_csv(tmp, index=False, float_format='%.10g'))


def write_json(obj, path):
    text = json.dumps(obj, indent=1, default=to_builtin, ensure_ascii=False)
    atomic(path, lambda tmp: tmp.write_text(text, encoding='utf-8'))


def fmt_time(seconds):
    """Format a duration as '12.3s', '4m 05s' or '1h 07m'."""
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    if minutes < 60:
        return f'{minutes}m {seconds:02d}s'
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m'


def load_inputs(folder):
    """config.json, dataset.parquet and edges.parquet of step 02, with the arrays every later step needs."""
    config = json.loads((folder / 'config.json').read_text(encoding='utf-8'))
    assert config['version'] == CONFIG_VERSION, f"config.json has version {config['version']}, this code expects {CONFIG_VERSION}"
    dataset = pd.read_parquet(folder / 'dataset.parquet')
    edges = pd.read_parquet(folder / 'edges.parquet')
    assert len(dataset) == config['files']['n_rows'] and len(edges) == config['files']['n_edges']
    y = dataset['y'].to_numpy(dtype=np.float64, na_value=np.nan)
    return SimpleNamespace(
        config=config, dataset=dataset,
        split=dataset['split'].to_numpy(), in_eval=dataset['in_eval'].to_numpy(dtype=bool), illicit=(y == 1),
        src=edges['src_row'].to_numpy(np.int64), dst=edges['dst_row'].to_numpy(np.int64))


def scale(values, spec):
    """One column of the profile `scaled` (config['transforms']): z = (t - mean) / std, with
    t = sign(x) log(1 + |x|) for the two log transformations and t = x otherwise (binary: mean 0, std 1)."""
    x = np.asarray(values, dtype=np.float64)
    if spec['transform'] in ('log1p+z', 'signed_log1p+z'):
        x = np.sign(x) * np.log1p(np.abs(x))
    return (x - spec['mean']) / spec['std']


def feature_matrix(data, menu, profile):
    """The input matrix of a menu for every node (float32): raw values (profile `raw`) or transformed
    (profile `scaled`). The menu `none` gives the constant input x = 1."""
    columns = data.config['menus'][menu]['columns']
    if not columns:
        return np.ones((len(data.dataset), 1), dtype=np.float32)
    X = data.dataset[columns].to_numpy(np.float64)
    if profile == 'scaled':
        specs = data.config['transforms']['profiles']['scaled']['columns']
        for k, col in enumerate(columns):
            X[:, k] = scale(X[:, k], specs[col])
    return X.astype(np.float32)


def eval_rows(data, split):
    """Row numbers of the evaluation rows of a split (labeled and with the Elliptic++ data)."""
    return np.flatnonzero((data.split == split) & data.in_eval)


def suggest_params(trial, space):
    """Ask an Optuna trial for one value of every hyper-parameter of a search space of config.json."""
    params = {}
    for name, spec in space.items():
        if spec['type'] == 'categorical':
            params[name] = trial.suggest_categorical(name, spec['choices'])
        elif spec['type'] == 'float':
            params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=spec['log'])
        elif spec['type'] == 'int':
            params[name] = trial.suggest_int(name, spec['low'], spec['high'], log=spec['log'])
        else:
            raise ValueError(f'unknown type {spec["type"]} for {name}')
    return params

In [6]:
# ==== SHARED CODE 2/4 · classical models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# XGBoost on the GPU with NumPy inputs prints this at every prediction; it is harmless.
warnings.filterwarnings('ignore', message='.*Falling back to prediction using DMatrix.*')


def fit_classical(model_name, params, fixed, X_train, y_train, X_val, y_val, seed, final=False):
    """Train Logistic Regression, Random Forest or XGBoost on the train rows.
    XGBoost stops adding trees when the validation PR-AUC stops improving (early stopping).
    `final=True` (step 04) also computes the out-of-bag score of the Random Forest."""
    fixed = dict(fixed)
    info = {}
    start = time.perf_counter()
    if model_name == 'logreg':
        model = LogisticRegression(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train)
    elif model_name == 'rf':
        oob = bool(fixed.pop('oob_score_in_final_runs', False)) and final
        model = RandomForestClassifier(**fixed, **params, oob_score=oob, random_state=seed)
        model.fit(X_train, y_train)
        if oob:
            info['oob_average_precision'] = float(average_precision_score(
                y_train, np.nan_to_num(model.oob_decision_function_[:, 1])))
    elif model_name == 'xgb':
        model = XGBClassifier(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        info['best_iteration'] = int(model.best_iteration)
    else:
        raise ValueError(model_name)
    info['seconds'] = time.perf_counter() - start
    scores = model.predict_proba(X_val)[:, 1]
    return SimpleNamespace(model=model, val_scores=scores, val_ap=float(average_precision_score(y_val, scores)), info=info)

In [7]:
# ==== SHARED CODE 3/4 · neural models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-gpu, and later 04-final-runs-gpu and 05-plots (to recompute embeddings).
# Change every copy together and raise SHARED_CODE_VERSION.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import ChebConv, GATv2Conv, GINConv, JumpingKnowledge, SAGEConv
from torch_geometric.utils import to_undirected

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def period_data(data, X, split, kind):
    """Tensors of one split on the GPU.
    kind 'graph' (GNNs): every node of the time steps of the split (unknown included), the edges among them made
    undirected; loss and metrics only on the evaluation rows (`mask`).
    kind 'rows' (MLP): only the evaluation rows of the split, no edges."""
    if kind == 'rows':
        rows = eval_rows(data, split)
        y = data.illicit[rows]
        return SimpleNamespace(
            x=torch.tensor(X[rows], device=DEVICE), edge_index=None,
            y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
            mask=torch.ones(len(rows), dtype=torch.bool, device=DEVICE), y_eval=y.astype(int), rows=rows)
    nodes = np.flatnonzero(data.split == split)
    local = np.full(len(data.split), -1)
    local[nodes] = np.arange(len(nodes))
    keep = local[data.src] >= 0
    assert (local[data.dst[keep]] >= 0).all(), 'an edge leaves the split'   # no edge links two time steps
    edge_index = torch.tensor(np.vstack([local[data.src[keep]], local[data.dst[keep]]]), dtype=torch.long)
    mask = data.in_eval[nodes]
    y = data.illicit[nodes]
    return SimpleNamespace(
        x=torch.tensor(X[nodes], device=DEVICE),
        edge_index=to_undirected(edge_index, num_nodes=len(nodes)).to(DEVICE),
        y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
        mask=torch.tensor(mask, device=DEVICE), y_eval=y[mask].astype(int), rows=nodes[mask])


class MLP(nn.Module):
    """`layers` hidden layers of width `hidden` (ReLU, dropout), then one output."""

    def __init__(self, d_in, hidden, layers, dropout):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        self.hidden = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.out = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index=None, return_hidden=False):
        for layer in self.hidden:
            x = F.dropout(F.relu(layer(x)), self.dropout, self.training)
        return x if return_hidden else self.out(x).squeeze(-1)


def make_conv(kind, d_in, d_out, args):
    if kind == 'sage':
        return SAGEConv(d_in, d_out, aggr=args['aggr'])
    if kind == 'gatv2':
        heads = args['heads'] if d_out > 1 else 1
        return GATv2Conv(d_in, d_out // heads, heads=heads, dropout=args['att_dropout'])
    if kind == 'cheb':
        return ChebConv(d_in, d_out, K=args['K'], normalization=args.get('normalization', 'sym'))
    if kind == 'gin':
        return GINConv(nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU(), nn.Linear(d_out, d_out)),
                       train_eps=args['train_eps'])
    raise ValueError(kind)


class GNN(nn.Module):
    """`layers` graph convolutions in one of three variants (Ferretti, D'Angelo and Ghini, 2025):
    'base' (the last convolution gives the output), 'lin' (a linear layer after the convolutions),
    'skip' (like 'lin', and the output of the first convolution is added to the output of the last one).
    `jumping_knowledge` ('max' or 'cat', GIN only) combines the outputs of every convolution before the linear layer."""

    def __init__(self, kind, d_in, hidden, layers, variant, dropout, jumping_knowledge=None, **conv_args):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        if variant == 'base':
            dims[-1] = 1
        self.convs = nn.ModuleList(make_conv(kind, a, b, conv_args) for a, b in zip(dims[:-1], dims[1:]))
        self.jk = None if jumping_knowledge is None else JumpingKnowledge(jumping_knowledge)
        width = hidden * layers if jumping_knowledge == 'cat' else hidden
        self.lin = None if variant == 'base' else nn.Linear(width, 1)
        self.variant, self.dropout = variant, dropout

    def forward(self, x, edge_index, return_hidden=False):
        outputs = []
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if self.lin is not None or i < len(self.convs) - 1:
                x = F.dropout(F.relu(x), self.dropout, self.training)
            outputs.append(x)
        if self.variant == 'skip':
            x = x + outputs[0]
        if self.jk is not None:
            x = self.jk(outputs)
        if return_hidden:
            return x
        if self.lin is not None:
            x = self.lin(x)
        return x.squeeze(-1)


def build_neural(model_name, params, d_in, fixed=None):
    """The network of a model from its hyper-parameters (lr and weight_decay belong to the optimizer)."""
    p = {k: v for k, v in params.items() if k not in ('lr', 'weight_decay')}
    fixed = dict(fixed or {})
    if model_name == 'mlp':
        net = MLP(d_in, **p)
    elif model_name == 'gatv2':
        heads, per_head = p.pop('heads'), p.pop('hidden_per_head')
        net = GNN('gatv2', d_in, hidden=heads * per_head, heads=heads, **p)
    elif model_name == 'gin':
        net = GNN('gin', d_in, variant='lin', **p)
    else:
        net = GNN(model_name, d_in, **p, **fixed)
    return net.to(DEVICE)


def predict(net, part):
    """Scores (probability of illicit) of the evaluation rows of one split."""
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(part.x, part.edge_index)[part.mask]).float().cpu().numpy()


def train_neural(net, train, val, lr, weight_decay, pos_weight, max_epochs, patience,
                 report=None, curves=False, train_ap_every=None, keep_best_state=False):
    """Full-batch training with Adam and binary cross-entropy weighted by `pos_weight`.
    After every epoch: PR-AUC on the validation rows; training stops after `patience` epochs without improvement,
    and the result is the best epoch. `report(epoch, best_ap)` is called after every epoch (pruning of the tuning).
    `curves` keeps one row per epoch (train and validation loss, validation PR-AUC, train PR-AUC every
    `train_ap_every` epochs); `keep_best_state` keeps a copy of the weights of the best epoch."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE))
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    best_ap, best_epoch, best_state, history, diverged = 0.0, 0, None, [], False
    start = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        net.train()
        optimizer.zero_grad()
        out = net(train.x, train.edge_index)
        loss = loss_fn(out[train.mask], train.y[train.mask])
        loss.backward()
        optimizer.step()
        net.eval()
        with torch.no_grad():
            val_out = net(val.x, val.edge_index)[val.mask]
        if not torch.isfinite(val_out).all():   # the training diverged (for example, a learning rate too high)
            diverged = True
            break
        val_ap = float(average_precision_score(val.y_eval, val_out.float().cpu().numpy()))
        if curves:
            row = {'epoch': epoch, 'train_loss': float(loss), 'val_loss': float(loss_fn(val_out, val.y[val.mask])),
                   'val_average_precision': val_ap}
            if train_ap_every and epoch % train_ap_every == 0:
                row['train_average_precision'] = float(average_precision_score(train.y_eval, predict(net, train)))
            history.append(row)
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            if keep_best_state:
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        if report is not None:
            report(epoch, best_ap)
        if epoch - best_epoch >= patience:
            break
    return {'best_ap': best_ap, 'best_epoch': best_epoch, 'epochs': epoch, 'diverged': diverged,
            'hit_max_epochs': (not diverged) and epoch == max_epochs and epoch - best_epoch < patience,
            'seconds': time.perf_counter() - start, 'n_params': sum(p.numel() for p in net.parameters()),
            'peak_gpu_mb': torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == 'cuda' else None,
            'curves': history, 'best_state': best_state}

In [8]:
# ==== SHARED CODE 4/4 · tuning driver · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import gc
import shutil
import warnings

import optuna
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore', category=optuna.exceptions.ExperimentalWarning)
FINISHED = (TrialState.COMPLETE, TrialState.PRUNED)   # the trials that count in the budget

# Expected seconds of one trial with (27, 192) columns: mean of the search space, pruning included
# (timing test of 24/10). Only used to weight the time estimate of the log.
EXPECTED_TRIAL_SECONDS = {'logreg': (0.5, 3.4), 'rf': (19, 120), 'xgb': (1.3, 4.9), 'mlp': (1.7, 1.7),
                          'sage': (5, 7), 'gatv2': (35, 37), 'cheb': (16, 24)}
NONE_MENU_SECONDS = {'cheb': 33, 'gin': 9.5}   # menu `none` (x = 1)


def expected_trial_seconds(model, n_columns):
    if n_columns == 0:
        return NONE_MENU_SECONDS[model]
    low, high = EXPECTED_TRIAL_SECONDS[model]
    width = min(max(n_columns, 27), 192)
    return low + (high - low) * (width - 27) / (192 - 27)


def budget_of(config, model, menu):
    trials = config['models'][model]['trials']
    return trials.get(menu, trials['default'])


def make_sampler(config, model, seed):
    """TPE for every model; the full grid for the Logistic Regression."""
    spec = config['models'][model]
    if spec.get('sampler') == 'grid':
        return optuna.samplers.GridSampler({name: s['choices'] for name, s in spec['space'].items()}, seed=seed)
    return optuna.samplers.TPESampler(seed=seed, n_startup_trials=config['tuning']['sampler']['n_startup_trials'])


def make_pruner(config, model):
    """Median pruning for the neural models; no pruning for the classical ones (one fit, nothing to compare)."""
    if model in config['training']['models']:
        p = config['tuning']['pruner']
        return optuna.pruners.MedianPruner(n_startup_trials=p['n_startup_trials'], n_warmup_steps=p['n_warmup_steps'])
    return optuna.pruners.NopPruner()


def free_memory():
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def guarded(objective):
    """Write the error of a failed trial (for example, GPU out of memory) in the trial itself, and free memory."""
    def run(trial):
        try:
            return objective(trial)
        except optuna.TrialPruned:
            raise
        except Exception as error:
            trial.set_user_attr('error', f'{type(error).__name__}: {error}'[:500])
            raise
        finally:
            free_memory()
    return run


def sync(local, drive):
    """Copy the local SQLite study to Drive (through a temporary file)."""
    if local.exists():
        tmp = drive.with_name(drive.name + '.tmp')
        shutil.copy(local, tmp)
        os.replace(tmp, drive)


class Progress:
    """One log line per trial: pairs done and left, time spent and an estimate of the time left.
    The estimate is weighted by the expected cost of each pair (EXPECTED_TRIAL_SECONDS) and corrected by the
    speed measured in this session. The lines are also appended to a log file on Drive."""

    def __init__(self, expected, trial_cost, finished, log_path):
        self.expected, self.trial_cost = expected, trial_cost   # {pair: expected seconds}, {pair: seconds per trial}
        self.finished, self.log_path = set(finished), log_path
        self.work_done, self.start, self.lines = 0.0, time.time(), []

    def eta(self, pair, fraction):
        left = sum(s for p, s in self.expected.items() if p not in self.finished) - self.expected[pair] * fraction
        elapsed = time.time() - self.start
        speed = elapsed / self.work_done if self.work_done > 0 else 1.0   # real seconds per expected second
        return elapsed, max(left, 0) * speed

    def pairs_text(self):
        return f'pairs done {len(self.finished)}, left {len(self.expected) - len(self.finished)}'

    def print(self, line):
        print(line, flush=True)
        self.lines.append(f'{datetime.now():%Y-%m-%d %H:%M:%S} {line}')

    def flush(self):
        if self.lines and self.log_path is not None:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write('\n'.join(self.lines) + '\n')
        self.lines = []


def tune_pair(study_name, pair, objective, sampler_factory, pruner, budget, progress, label, dirs, seed):
    """Run trials until `budget` of them are finished (complete or pruned). The Optuna study lives in SQLite on the
    local disk and is copied to Drive at most every SYNC_SECONDS and at the end; if a copy is on Drive, the study
    restarts from it. A trial left half-done by a crash is marked as failed (heartbeat) and not counted.
    Resume rule: the sampler seed is `seed` + the number of trials already finished."""
    local_db, drive_db = dirs.local / f'{study_name}.db', dirs.studies / f'{study_name}.db'
    dirs.local.mkdir(parents=True, exist_ok=True)
    dirs.studies.mkdir(parents=True, exist_ok=True)
    if drive_db.exists() and not local_db.exists():
        shutil.copy(drive_db, local_db)
    storage = optuna.storages.RDBStorage(f'sqlite:///{local_db}', heartbeat_interval=60, grace_period=120)
    study = optuna.create_study(study_name=study_name, storage=storage, direction='maximize',
                                load_if_exists=True, pruner=pruner)
    n_finished = len(study.get_trials(deepcopy=False, states=FINISHED))
    study.sampler = sampler_factory(seed + n_finished)
    if n_finished:
        progress.print(f'{label} restarting from {n_finished} finished trials')
    last_sync, failures = time.time(), 0
    try:
        while n_finished < budget:
            study.optimize(guarded(objective), n_trials=1, catch=(Exception,))
            trial = study.trials[-1]
            if trial.state == TrialState.FAIL:
                failures += 1
                if failures >= MAX_FAILED_IN_A_ROW:
                    raise RuntimeError(f'{study_name}: {failures} failed trials in a row. '
                                       f'Last error: {trial.user_attrs.get("error")}')
            else:
                failures, n_finished = 0, n_finished + 1
                progress.work_done += progress.trial_cost[pair]
            complete = study.get_trials(deepcopy=False, states=(TrialState.COMPLETE,))
            best = max((t.value for t in complete), default=float('nan'))
            elapsed, left = progress.eta(pair, n_finished / budget)
            progress.print(f'{label} trial {n_finished}/{budget} {trial.state.name.lower()} · best val PR-AUC {best:.4f} · '
                           f'{progress.pairs_text()} · elapsed {fmt_time(elapsed)}, left ~{fmt_time(left)}')
            if time.time() - last_sync >= SYNC_SECONDS:
                sync(local_db, drive_db)
                progress.flush()
                last_sync = time.time()
    finally:
        sync(local_db, drive_db)
        progress.flush()
    return study


def classical_objective(data, model, menu):
    """Optuna objective of LogReg, RF or XGBoost on one menu: validation PR-AUC of one fit."""
    spec = data.config['models'][model]
    X = feature_matrix(data, menu, spec['profile'])
    tr, va = eval_rows(data, 'train'), eval_rows(data, 'val')
    X_tr, y_tr, X_va, y_va = X[tr], data.illicit[tr].astype(int), X[va], data.illicit[va].astype(int)
    del X

    def objective(trial):
        params = suggest_params(trial, spec['space'])
        set_seed(TUNING_SEED)
        result = fit_classical(model, params, spec.get('fixed', {}), X_tr, y_tr, X_va, y_va, seed=TUNING_SEED)
        for key, value in result.info.items():
            trial.set_user_attr(key, value)
        return result.val_ap
    return objective


def neural_objective(data, model, menu, max_epochs):
    """Optuna objective of MLP and GNNs on one menu: best validation PR-AUC of one training with early stopping.
    After every epoch the best value so far is reported to the pruner."""
    config = data.config
    spec = config['models'][model]
    X = feature_matrix(data, menu, spec['profile'])
    kind = 'rows' if model == 'mlp' else 'graph'
    train, val = period_data(data, X, 'train', kind), period_data(data, X, 'val', kind)
    del X
    patience = config['training']['patience']['mlp' if model == 'mlp' else 'gnn']
    pos_weight = config['labels']['pos_weight']

    def objective(trial):
        params = suggest_params(trial, spec['space'])
        set_seed(TUNING_SEED)
        net = build_neural(model, params, train.x.shape[1], spec.get('fixed', {}))

        def report(epoch, best_ap):
            trial.report(best_ap, epoch)
            if trial.should_prune():
                trial.set_user_attr('epochs', epoch)
                raise optuna.TrialPruned()

        result = train_neural(net, train, val, params['lr'], params['weight_decay'], pos_weight, max_epochs, patience,
                              report=report)
        for key in ['best_epoch', 'epochs', 'hit_max_epochs', 'diverged', 'seconds', 'n_params', 'peak_gpu_mb']:
            trial.set_user_attr(key, result[key])
        return result['best_ap']
    return objective


def tune_one_pair(data, menu, model, label, progress, dirs):
    """Tune one (menu, model) pair. A neural pair whose best trial stops at the epoch cap is tuned again with the
    larger cap of config.json. Writes best/<pair>.json and trials/<pair>.csv."""
    config = data.config
    pair, spec = f'{menu}__{model}', config['models'][model]
    budget = budget_of(config, model, menu)
    neural = model in config['training']['models']
    max_epochs = config['training']['max_epochs']
    while True:
        study_name = pair if max_epochs == config['training']['max_epochs'] else f'{pair}__max{max_epochs}'
        objective = neural_objective(data, model, menu, max_epochs) if neural else classical_objective(data, model, menu)
        study = tune_pair(study_name, pair, objective, lambda s: make_sampler(config, model, s), make_pruner(config, model),
                          budget, progress, label, dirs, TUNING_SEED)
        del objective
        free_memory()
        best = study.best_trial
        if neural and best.user_attrs.get('hit_max_epochs') and max_epochs < config['training']['max_epochs_if_capped']:
            max_epochs = config['training']['max_epochs_if_capped']
            progress.expected[pair] *= 2
            progress.print(f'{label} the best trial stopped at the epoch cap: tuning again with max_epochs = {max_epochs}')
            continue
        break
    states = [t.state for t in study.trials]
    record = {
        'pair': pair, 'menu': menu, 'model': model,
        'params': best.params, 'fixed': spec.get('fixed', {}),
        'max_epochs': max_epochs if neural else None,
        'patience': config['training']['patience']['mlp' if model == 'mlp' else 'gnn'] if neural else None,
        'val_average_precision': best.value, 'best_trial': best.number, 'best_trial_info': best.user_attrs,
        'budget': budget, 'trials': {s.name: states.count(s) for s in (TrialState.COMPLETE, TrialState.PRUNED, TrialState.FAIL)},
        'tuning_seconds': sum(t.duration.total_seconds() for t in study.trials if t.duration is not None),
        'study': study_name, 'tuning_seed': TUNING_SEED, 'shared_code_version': SHARED_CODE_VERSION,
        'config_version': config['version'], 'notebook': NOTEBOOK,
        'created': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }
    write_csv(study.trials_dataframe(), dirs.trials / f'{pair}.csv')
    write_json(record, dirs.best / f'{pair}.json')   # written last: its presence means "pair finished"
    return record


def run_tuning(data, models, dirs, only_pairs=None):
    """Tune every pair of `models`, skipping the pairs whose best/<pair>.json is already on Drive."""
    config = data.config
    pairs = [(menu, model) for menu, model in config['pairs']
             if model in models and (only_pairs is None or f'{menu}__{model}' in only_pairs)]
    cost = {f'{m}__{mo}': expected_trial_seconds(mo, config['menus'][m]['n_columns']) for m, mo in pairs}
    expected = {p: budget_of(config, p.split('__')[1], p.split('__')[0]) * c for p, c in cost.items()}
    finished = [p for p in expected if (dirs.best / f'{p}.json').exists()]
    progress = Progress(expected, cost, finished, dirs.logs / f'03-tuning-{NOTEBOOK}.log')
    total = sum(expected.values())
    progress.print(f'{len(pairs)} pairs in this notebook, {len(finished)} already done. '
                   f'Expected time of the missing ones: ~{fmt_time(sum(expected[p] for p in expected if p not in finished))} '
                   f'(of ~{fmt_time(total)}).')
    for k, (menu, model) in enumerate(pairs, 1):
        pair = f'{menu}__{model}'
        label = f'[pair {k}/{len(pairs)} · {model} × {menu}]'
        if pair in progress.finished:
            saved = json.loads((dirs.best / f'{pair}.json').read_text(encoding='utf-8'))
            note = '' if saved['shared_code_version'] == SHARED_CODE_VERSION else \
                f" (WARNING: tuned with shared code version {saved['shared_code_version']})"
            progress.print(f"{label} already done: val PR-AUC {saved['val_average_precision']:.4f}{note}")
            continue
        record = tune_one_pair(data, menu, model, label, progress, dirs)
        progress.finished.add(pair)
        progress.print(f"{label} done: best val PR-AUC {record['val_average_precision']:.4f} "
                       f"(trial {record['best_trial']}, {record['trials']}), best/{pair}.json written")
    progress.flush()


def write_summary(dirs, path):
    """One row per finished pair (of both tuning notebooks), from the best/*.json files."""
    rows = []
    for file in sorted(dirs.best.glob('*.json')):
        r = json.loads(file.read_text(encoding='utf-8'))
        rows.append({'menu': r['menu'], 'model': r['model'], 'val_average_precision': r['val_average_precision'],
                     'complete': r['trials']['COMPLETE'], 'pruned': r['trials']['PRUNED'], 'failed': r['trials']['FAIL'],
                     'best_epoch': r['best_trial_info'].get('best_epoch'), 'max_epochs': r['max_epochs'],
                     'hit_max_epochs': r['best_trial_info'].get('hit_max_epochs'),
                     'tuning_minutes': r['tuning_seconds'] / 60, 'shared_code_version': r['shared_code_version'],
                     'params': json.dumps(r['params'])})
    summary = pd.DataFrame(rows)
    if len(summary):
        write_csv(summary, path)
    return summary

## 2. Load the inputs

`config.json` decides everything: menus, transformations, search spaces, numbers of trials, seeds. The checks: the files of step 02 have the expected version and sizes, and a GPU is available.

In [9]:
DIRS = SimpleNamespace(best=TUNING_DIR / 'best', trials=TUNING_DIR / 'trials', studies=TUNING_DIR / 'studies',
                       logs=TUNING_DIR / 'logs', local=LOCAL_STUDIES)
start = time.time()
data = load_inputs(LOCAL_INPUT)
TUNING_SEED = data.config['seeds']['tuning']
print(f'Inputs loaded in {fmt_time(time.time() - start)}: {len(data.dataset):,} rows, '
      f'config version {data.config["version"]}, tuning seed {TUNING_SEED}.')
for split in ['train', 'val', 'test']:
    rows = eval_rows(data, split)
    print(f'{split}: {len(rows):,} evaluation rows, {data.illicit[rows].sum():,} illicit')

assert DEVICE.type == 'cuda', 'No GPU: choose Runtime -> Change runtime type -> T4 GPU, then run again.'
print('GPU:', torch.cuda.get_device_name(0))

if FORCE_RECOMPUTE:
    for menu, model in data.config['pairs']:
        if model in MODELS:
            for path in [DIRS.best / f'{menu}__{model}.json', DIRS.trials / f'{menu}__{model}.csv',
                         *DIRS.studies.glob(f'{menu}__{model}*.db'), *LOCAL_STUDIES.glob(f'{menu}__{model}*.db')]:
                path.unlink(missing_ok=True)

Inputs loaded in 0.6s: 203,769 rows, config version 1, tuning seed 42.
train: 26,750 evaluation rows, 2,954 illicit
val: 9,429 evaluation rows, 1,067 illicit
test: 9,866 evaluation rows, 524 illicit
GPU: Tesla T4


## 3. Tests on toy problems

Before the long runs, a few quick checks with known answers.
- **XGBoost** learns a toy table where one column carries the label, and the same seed gives the same scores.
- **Message passing**: on a toy graph of pairs of nodes, the label of a node depends only on the feature of its partner. An MLP, which sees only the node itself, cannot learn it; a GraphSAGE, which also sees the neighbour, must learn it. Every model and variant is also built and trained for a few epochs.
- **Tuning driver**: a crash in the middle of a study, then a restart from the Drive copy with an empty local disk; and the full grid of the Logistic Regression.

In [10]:
rng = np.random.default_rng(0)
toy_y = (rng.random(1000) < 0.1).astype(int)
toy_X = np.column_stack([toy_y + 0.3 * rng.standard_normal(1000), rng.standard_normal(1000)]).astype(np.float32)
tr, va = np.arange(700), np.arange(700, 1000)
toy_params = {'xgb': {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1.0, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 1.0, 'gamma': 0.0, 'scale_pos_weight': 1.0}}
for model, params in toy_params.items():
    fixed = data.config['models'][model].get('fixed', {})
    runs = [fit_classical(model, params, fixed, toy_X[tr], toy_y[tr], toy_X[va], toy_y[va], seed=TUNING_SEED)
            for _ in range(2)]
    # prevalence 0.1: a model that learns nothing gets about 0.1. The bar is well above it but not tight,
    # because a fully grown Random Forest with one pure-noise column gives coarser scores than a linear model.
    assert runs[0].val_ap > 0.7, (model, runs[0].val_ap)
    assert np.allclose(runs[0].val_scores, runs[1].val_scores, atol=1e-6), f'{model}: same seed, different scores'
    print(f'{model}: toy validation PR-AUC {runs[0].val_ap:.3f}, reproducible')

xgb: toy validation PR-AUC 0.902, reproducible


In [11]:
def toy_graph(seed, n=600):
    """n pairs of nodes u - v. Only the v nodes are evaluated, and v is illicit when its partner u has feature > 1:
    the label of v depends only on its neighbour."""
    rng = np.random.default_rng(seed)
    f_u, f_v = rng.standard_normal(n), rng.standard_normal(n)
    y = np.concatenate([np.zeros(n), (f_u > 1).astype(float)])
    mask = np.concatenate([np.zeros(n, dtype=bool), np.ones(n, dtype=bool)])
    edge_index = torch.tensor(np.vstack([np.arange(n), np.arange(n) + n]), dtype=torch.long)
    graph = SimpleNamespace(
        x=torch.tensor(np.concatenate([f_u, f_v])[:, None], dtype=torch.float32, device=DEVICE),
        edge_index=to_undirected(edge_index, num_nodes=2 * n).to(DEVICE),
        y=torch.tensor(y, dtype=torch.float32, device=DEVICE), mask=torch.tensor(mask, device=DEVICE),
        y_eval=y[mask].astype(int))
    rows = SimpleNamespace(x=graph.x[mask], edge_index=None, y=graph.y[mask],
                           mask=torch.ones(n, dtype=torch.bool, device=DEVICE), y_eval=graph.y_eval)
    return graph, rows


(toy_train_graph, toy_train_rows), (toy_val_graph, toy_val_rows) = toy_graph(0), toy_graph(1)
toy_pos_weight = float((toy_train_graph.y_eval == 0).sum() / toy_train_graph.y_eval.sum())
results = {}
for model, params, train_part, val_part in [
        ('mlp', {'layers': 2, 'hidden': 32, 'dropout': 0.0}, toy_train_rows, toy_val_rows),
        ('sage', {'layers': 2, 'hidden': 32, 'aggr': 'mean', 'variant': 'lin', 'dropout': 0.0}, toy_train_graph, toy_val_graph)]:
    aps = []
    for _ in range(2):
        set_seed(TUNING_SEED)
        net = build_neural(model, params, 1)
        aps.append(train_neural(net, train_part, val_part, 1e-2, 0.0, toy_pos_weight, 300, 100)['best_ap'])
    results[model] = aps[0]
    print(f'{model}: toy validation PR-AUC {aps[0]:.3f} (same seed again: {aps[1]:.3f})')
    assert abs(aps[0] - aps[1]) < 0.02, f'{model}: same seed, very different results'
assert results['sage'] > 0.9, 'GraphSAGE did not learn from the neighbour'
assert results['mlp'] < 0.6, 'the MLP should not be able to learn this task'

# every model and variant builds, trains a few epochs and gives finite scores
spaces = data.config['models']
for model in ['sage', 'gatv2', 'cheb', 'gin']:
    base = {name: (s['choices'][0] if s['type'] == 'categorical' else s['low']) for name, s in spaces[model]['space'].items()}
    options = [{'variant': v} for v in ['base', 'lin', 'skip']] if model != 'gin' else \
        [{'jumping_knowledge': jk} for jk in [None, 'max', 'cat']]
    for option in options:
        net = build_neural(model, {**base, **option}, 1, spaces[model].get('fixed', {}))
        result = train_neural(net, toy_train_graph, toy_val_graph, 1e-3, 0.0, toy_pos_weight, 3, 100)
        assert np.isfinite(predict(net, toy_val_graph)).all() and result['epochs'] == 3, (model, option)
print('Every model and variant runs.')

mlp: toy validation PR-AUC 0.191 (same seed again: 0.191)
sage: toy validation PR-AUC 0.997 (same seed again: 0.997)
Every model and variant runs.


In [12]:
# Test of the tuning driver on a toy objective, in a temporary folder (nothing is written to Drive).
# 1) A crash at trial 3 (like a stopped cell) is marked as failed; the study is restarted from its Drive copy
#    with an empty local disk (like a new Colab session) and reaches its budget of 6 finished trials.
# 2) The grid of the Logistic Regression (from config.json, with None among the choices) gives 20 different trials.
import tempfile
from pathlib import Path


class SimulatedCrash(KeyboardInterrupt):
    pass


def toy_objective(crash_at=None):
    def objective(trial):
        x = trial.suggest_float('x', -2.0, 2.0)
        if trial.number == crash_at:
            raise SimulatedCrash()
        return -(x - 0.5) ** 2
    return objective


toy_root = Path(tempfile.mkdtemp())
toy_dirs = SimpleNamespace(local=toy_root / 'local', studies=toy_root / 'drive')
toy_progress = Progress({'toy': 6.0}, {'toy': 1.0}, [], None)
toy_sampler = lambda seed: optuna.samplers.TPESampler(seed=seed)
try:
    tune_pair('toy', 'toy', toy_objective(crash_at=3), toy_sampler, optuna.pruners.NopPruner(), 6, toy_progress,
              '[toy]', toy_dirs, TUNING_SEED)
    raise AssertionError('the simulated crash did not stop the study')
except SimulatedCrash:
    print('crash simulated at trial 3')
(toy_dirs.local / 'toy.db').unlink()   # a new Colab session starts with an empty local disk
study = tune_pair('toy', 'toy', toy_objective(), toy_sampler, optuna.pruners.NopPruner(), 6, toy_progress,
                  '[toy]', toy_dirs, TUNING_SEED)
states = [t.state for t in study.trials]
assert states.count(TrialState.COMPLETE) == 6 and states.count(TrialState.FAIL) == 1, states
assert study.best_trial.value > -0.5

grid = optuna.create_study(direction='maximize', sampler=make_sampler(data.config, 'logreg', TUNING_SEED))
logreg_space = data.config['models']['logreg']['space']
grid.optimize(lambda trial: suggest_params(trial, logreg_space)['C'], n_trials=20)
assert len({(t.params['C'], t.params['class_weight']) for t in grid.trials}) == 20
shutil.rmtree(toy_root, ignore_errors=True)
print('Driver test passed: resume after a crash and full grid.')

[toy] trial 1/6 complete · best val PR-AUC -1.0037 · pairs done 0, left 1 · elapsed 2.0s, left ~9.9s
[toy] trial 2/6 complete · best val PR-AUC -1.0037 · pairs done 0, left 1 · elapsed 2.3s, left ~4.5s
[toy] trial 3/6 complete · best val PR-AUC -0.1832 · pairs done 0, left 1 · elapsed 2.6s, left ~2.6s


[W 2026-10-25 07:42:04,596] Trial 3 failed with parameters: {'x': 0.3946339367881464} because of the following error: SimulatedCrash().
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_5188/2378030640.py", line 63, in run
    return objective(trial)
  File "/tmp/ipykernel_5188/2031093301.py", line 17, in objective
    raise SimulatedCrash()
SimulatedCrash
[W 2026-10-25 07:42:04,598] Trial 3 failed with value None.


crash simulated at trial 3
[toy] restarting from 3 finished trials
[toy] trial 4/6 complete · best val PR-AUC -0.1832 · pairs done 0, left 1 · elapsed 3.3s, left ~1.6s
[toy] trial 5/6 complete · best val PR-AUC -0.0911 · pairs done 0, left 1 · elapsed 3.6s, left ~0.7s
[toy] trial 6/6 complete · best val PR-AUC -0.0911 · pairs done 0, left 1 · elapsed 3.9s, left ~0.0s
Driver test passed: resume after a crash and full grid.


## 4. Tuning

One log line per trial, for example

```
[pair 7/32 · gatv2 × graph] trial 23/40 complete · best val PR-AUC 0.6120 · pairs done 6, left 26 · elapsed 1h 10m, left ~3h 20m
```

The estimate of the time left is weighted by the expected cost of each pair (from the timing test) and corrected by the speed measured in this session, so it becomes more reliable after the first pairs. If Colab disconnects, run the notebook again from the top: the finished pairs are skipped and the running one restarts from its last copy on Drive.

In [13]:
run_tuning(data, MODELS, DIRS, ONLY_PAIRS)

32 pairs in this notebook, 23 already done. Expected time of the missing ones: ~1h 54m (of ~4h 33m).
[pair 1/32 · xgb × local] already done: val PR-AUC 0.9576
[pair 2/32 · xgb × local_agg] already done: val PR-AUC 0.9620
[pair 3/32 · xgb × graph] already done: val PR-AUC 0.3672
[pair 4/32 · xgb × econ] already done: val PR-AUC 0.8201
[pair 5/32 · xgb × transparent] already done: val PR-AUC 0.8166
[pair 6/32 · xgb × local_agg_graph] already done: val PR-AUC 0.9624
[pair 7/32 · mlp × local] already done: val PR-AUC 0.8974
[pair 8/32 · mlp × local_agg] already done: val PR-AUC 0.8386
[pair 9/32 · mlp × graph] already done: val PR-AUC 0.4168
[pair 10/32 · mlp × econ] already done: val PR-AUC 0.6986
[pair 11/32 · mlp × transparent] already done: val PR-AUC 0.7575
[pair 12/32 · mlp × local_agg_graph] already done: val PR-AUC 0.7820
[pair 13/32 · sage × local] already done: val PR-AUC 0.9233
[pair 14/32 · sage × local_agg] already done: val PR-AUC 0.9096
[pair 15/32 · sage × graph] already do

## 5. Summary

`summary.csv` has one row per finished pair, of both tuning notebooks. The table below shows the best validation PR-AUC of each pair (rows: models, columns: menus).

In [14]:
summary = write_summary(DIRS, TUNING_DIR / 'summary.csv')
mine = [f'{menu}__{model}' for menu, model in data.config['pairs'] if model in MODELS]
done = [p for p in mine if (DIRS.best / f'{p}.json').exists()]
print(f'Pairs of this notebook finished: {len(done)} of {len(mine)}.')
if len(summary):
    order = list(data.config['menus'])
    table = summary.pivot(index='model', columns='menu', values='val_average_precision')
    display(table.reindex(columns=[m for m in order if m in table.columns]).round(4))

Pairs of this notebook finished: 32 of 32.


menu,local,local_agg,graph,econ,transparent,local_agg_graph,none
model,,,,,,,
cheb,0.9297,0.8363,0.4741,0.7268,0.6834,0.8325,0.3637
gatv2,0.8990,0.8463,0.4770,0.6854,0.7291,0.8714,NaN
gin,NaN,NaN,NaN,NaN,NaN,NaN,0.2803
logreg,0.5237,0.7114,0.2437,0.5402,0.5620,0.7259,NaN
mlp,0.8974,0.8386,0.4168,0.6986,0.7575,0.7820,NaN
rf,0.9545,0.9580,0.3544,0.7250,0.7459,0.9570,NaN
sage,0.9233,0.9096,0.4924,0.7638,0.6755,0.8940,NaN
xgb,0.9576,0.9620,0.3672,0.8201,0.8166,0.9624,NaN


In [15]:
drive.flush_and_unmount()
print('Every file is on Drive in', TUNING_DIR)

Every file is on Drive in /content/drive/MyDrive/UNIVERSITA/block-crypto/data/03-tuning
